# Realizing our position

This notebook covers understanding our (Earth's) position in the Solar System using SpiceyPy. As a heads-up, SpiceyPy is a wrapper around CSpice, an API for the Spice library maintained by NASA and JPL. The Spice library uses data sources from previous observations (Kernels) and lets us compute, not just past records but present and the future with that data!

See also:
1. [The Spice Toolkit](https://naif.jpl.nasa.gov/naif/toolkit.html)
2. [SpiceyPy](https://spiceypy.readthedocs.io/en/stable/)
3. [Kernel Repository](https://naif.jpl.nasa.gov/pub/naif/)

## Importing requirements

In [1]:
import spiceypy
import datetime
import math

## Loading the required Kernels

1. lsk/naif0012 refers to the kernel maintaining leapseconds required to measure Ehemeris Time (ET), see [README](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/aareadme.txt)
2. spk/de432s.bsp refers to the kernel maintaining planetary observations for planets in our solar system, capable of making calculations between 1949 and 2050. See [SUMMARY](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/aa_summaries.txt) 

In [2]:
spiceypy.furnsh("../kernels/lsk/naif0012.tls")
spiceypy.furnsh("../kernels/spk/de432s.bsp")

## Time to compute for

1. Spice requires any time related fields to be in the UTC format
2. To run computations, Spice uses Ephemeris Time (ET)

Ephemeris Time (ET) represents the number of seconds that have passed since a principle point in time. (I still need a better understanding here)

In [3]:
date_today = datetime.datetime.today().strftime(r"%Y-%m-%dT00:00:00")

print(date_today)

2026-02-27T00:00:00


In [4]:
ephimeris_time = spiceypy.utc2et(date_today)

print(f"{ephimeris_time}s")

825422469.1853445s


## State of Earth at a given Ephimeris Time

The `spkgeo` function returns a 'state vecotor' with respect to an observer, as well as time taken by light to reach the target from the observer in seconds. The state vector indicates the x, y and z positions on a plane of reference. Again, these coordinates are with respect to the observer and differ from one plane of reference to another.

In our case, we select the target to be the Earth's physical center, deoted by id 399.  
We set the ephimeris time to be 00:00 Hours UTC, Today.  
Our plane of reference is ECLIPJ2000 (Solar System as observed from top).  
And our observer is the Sun, denoted by id 10.  

In [5]:
earth_state_vector, light_time = spiceypy.spkgeo(
    targ=399,
    et=ephimeris_time,
    ref="ECLIPJ2000",
    obs=10
)

print(f"(X, Y, Z) = ({earth_state_vector[0], earth_state_vector[1], earth_state_vector[2]}) with respect to ECLIPJ2000")

print(f"Time taken by light from Sun to reach the Earth is {light_time / 60.0} minutes")

(X, Y, Z) = ((np.float64(-137386867.82551777), np.float64(55407853.364737645), np.float64(-2911.8786457441747))) with respect to ECLIPJ2000
Time taken by light from Sun to reach the Earth is 8.235645360184508 minutes


To understand the actual/physical distance between Sun and Earth, we can use calculate the magnitude of the vector formed by the three coordinates

In [6]:
earth_sun_distance = math.sqrt(
    earth_state_vector[0] ** 2.0
    + earth_state_vector[1] ** 2.0
    + earth_state_vector[2] ** 2.0
)

print(f"The distance between Earth and Sun is {earth_sun_distance} km")

The distance between Earth and Sun is 148139061.94476053 km


An astronomical unit is the average distance between Earth and the Sun. Hence, 1 AU describes the average distance between the Earth and the Sun

In [7]:
earth_sun_distance_in_au = spiceypy.convrt(earth_sun_distance, "km", "au")
print(f"The distance between Earth and Sun is {earth_sun_distance_in_au} AU")

The distance between Earth and Sun is 0.9902484663522019 AU
